### 12.1 pd与模型代码的接口  


机器学习->核心->特征工程  
特征工程方法->groupby  
提取好工程

In [1]:
import pandas as pd
import numpy as np

In [3]:
# pd与其他分析库通常是靠Numpy的数据结合起来的。
# 将DF转化为np数组，使用to_numpy方法
data = pd.DataFrame({"x0": [1, 2, 3, 4, 5],
                     "x1": [0.01, -0.01, 0.25, -4.1, 0.],
                     "y": [-1.5, 0., 3.6, 1.3, -2.]})

data

,x0,x1,y
0,1,0.01,-1.5
1,2,-0.01,0.0
2,3,0.25,3.6
3,4,-4.10,1.3
4,5,0.00,-2.0


In [5]:
data.columns

Index(['x0', 'x1', 'y'], dtype='object')

In [5]:
data.to_numpy()

array([[ 1.  ,  0.01, -1.5 ],
       [ 2.  , -0.01,  0.  ],
       [ 3.  ,  0.25,  3.6 ],
       [ 4.  , -4.1 ,  1.3 ],
       [ 5.  ,  0.  , -2.  ]])

In [7]:
# 转换回DF，传入一个二维ndarray(可带列名)：
df2 = pd.DataFrame(data.to_numpy(), columns=["one", "two", "three"])
df2

,one,two,three
0,1.0,0.01,-1.5
1,2.0,-0.01,0.0
2,3.0,0.25,3.6
3,4.0,-4.10,1.3
4,5.0,0.00,-2.0


to_numpy方法一般用于同构化数据。例如数据全是数值类型。  

如果数据是异构化的，结果会是py对象的ndarray

In [10]:
df3 = data.copy()
df3["strings"] = ["a", "b", "c", "d", "e"]
df3

,x0,x1,y,strings
0,1,0.01,-1.5,a
1,2,-0.01,0.0,b
2,3,0.25,3.6,c
3,4,-4.10,1.3,d
4,5,0.00,-2.0,e


In [12]:
df3.to_numpy()

array([[1, 0.01, -1.5, 'a'],
       [2, -0.01, 0.0, 'b'],
       [3, 0.25, 3.6, 'c'],
       [4, -4.1, 1.3, 'd'],
       [5, 0.0, -2.0, 'e']], dtype=object)

In [14]:
# 对于一些模型只使用列的子集，loc和iloc、to_numpy：
model_cols = ["x0", "x1"]
data.loc[:, model_cols].to_numpy()  # 将子列x0和x1转为numpy

array([[ 1.  ,  0.01],
       [ 2.  , -0.01],
       [ 3.  ,  0.25],
       [ 4.  , -4.1 ],
       [ 5.  ,  0.  ]])

一些库原生支持pandas，会自动完成这些工作：从DataFrame转换为NumPy，将模型参数名添加到输出表的列或Series上。对于其他情况，你可以手工进行“元数据管理”。

**在7.5节中，我们学习了pandas的Categorical类型和pandas.get_dummies函数**   
假设数据集中有一个非数值列：  

In [18]:
data["category"] = pd.Categorical(["a", "b", "a", "a", "b"],
                                  categories=["a", "b"])
data

,x0,x1,y,category
0,1,0.01,-1.5,a
1,2,-0.01,0.0,b
2,3,0.25,3.6,a
3,4,-4.10,1.3,a
4,5,0.00,-2.0,b


In [20]:
# 如果我们想将'category'列替换为虚拟变量，可以创建虚拟变量
# 删除'category'列，然后连接到结果中：
dummies = pd.get_dummies(data.category, prefix='category').astype(int)  # 创建机器学习的二进制变量
data_with_dummies = data.drop("category", axis=1).join(dummies)
data_with_dummies

,x0,x1,y,category_a,category_b
0,1,0.01,-1.5,1,0
1,2,-0.01,0.0,0,1
2,3,0.25,3.6,1,0
3,4,-4.10,1.3,1,0
4,5,0.00,-2.0,0,1


将分类变量category转换为机器学习模型能够理解的二进制形式，然后将这些二进制列与原始数据集结合，形成一个新的数据集data_with_dummies。这样就可以在不损失信息的前提下，将分类数据输入到模型中。



**prefix='category'：**为生成的新列添加前缀，使生成的列名更具可读性。例如，如果有类别"A"、"B"、"C"，生成的列可能是category_A, category_B, category_C。




**data.drop("category", axis=1)：**从原始数据集中删除category这一列，因为它已经被转换成多个二进制列，原来的分类列不再需要。  
axis=1：表示删除的是列而不是行。  



**.join(dummies)**  
data.drop("category", axis=1).join(dummies)：将原始数据（删除了category列后）与生成的二进制哑变量列进行合并，形成一个新的DataFrame。

用虚拟变量拟合某些统计模型会有一些细微差别。当你不只有数值列时，使用**Patsy**更简单，且更不容易出错。

patsy是"公式语法"用于描述统计模型，尤其是线性模型，受R和S统计编程语言公式语法的启发，但不完全一致

### 12.2 用Patsy创建模型描述

patsy的特殊用法：
y ~ x0 + x1  


a+b不是将a与b相加的意思，而是为模型创建设计矩阵使用的术语。



patsy.dmatrices函数接收一个公式字符串和一个数据集（可以是DataFrame或数组字典），为线性模型创建设计矩阵：

In [66]:
data = pd.DataFrame({"x0": [1, 2, 3, 4, 5],
                     "x1": [0.01, -0.01, 0.25, -4.1, 0.],
                     "y": [-1.5, 0., 3.6, 1.3, -2.]})

data

,x0,x1,y
0,1,0.01,-1.5
1,2,-0.01,0.0
2,3,0.25,3.6
3,4,-4.10,1.3
4,5,0.00,-2.0


patsy.dmatrices用于将公式形式的线性模型表达式转换为设计矩阵（design matrix），这是在进行统计建模或回归分析时非常常用的步骤。



Patsy 提供了一种方便的方式，通过公式语言描述因变量和自变量，并将其转换为用于回归分析的矩阵形式。

In [69]:
import patsy

y, X = patsy.dmatrices("y ~ x0 + x1", data)

y  # 5行一列的矩阵

DesignMatrix with shape (5, 1)
     y
  -1.5
   0.0
   3.6
   1.3
  -2.0
  Terms:
    'y' (column 0)

In [31]:
X  # 5行三列

DesignMatrix with shape (5, 3)
  Intercept  x0     x1
          1   1   0.01
          1   2  -0.01
          1   3   0.25
          1   4  -4.10
          1   5   0.00
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'x1' (column 2)

设计矩阵 X 的形状是 (5, 3)，表示它有 5 行（对应于 5 个观测数据）和 3 列（包括截距项、x0 和 x1 变量）。

截距项（也称为常数项、偏置项）是在回归分析中用于表示回归模型的偏移量。它允许模型在不依赖于自变量（x0, x1 等）的情况下有一个基线值或起点。即使所有的自变量为 0，模型的预测值也不会是 0，而是等于这个截距项。

截距项的来源：  
在公式 y ~ x0 + x1 中，Patsy 默认会自动为模型添加一个截距项（常数项）。在 patsy.dmatrices() 中，截距项对应设计矩阵中的第一列，全为 1。其目的是通过添加一个常数项，使模型能够有一个不依赖于输入特征的基线输出值。

在回归模型 y = β0 + β1 * x0 + β2 * x1 中：  

β0：就是截距项（常数项），它不依赖于 x0 或 x1，是模型的基线预测值。  
β1, β2：是自变量 x0 和 x1 对应的回归系数。  




当你调用 patsy.dmatrices("y ~ x0 + x1", data) 时：  

y ~ x0 + x1：表示 y 是因变量，x0 和 x1 是自变量。  
Patsy 自动添加了一个截距项，对应设计矩阵 X 的第一列为 1，即每个观测数据的截距项为 1。  

如果你希望不添加截距项，可以在公式中使用 0 + 来明确表示不包含截距项。



y, X = patsy.dmatrices("y ~ 0 + x0 + x1", data)


In [71]:
# 这些 Patsy 的 DesignMatrix 实例是 NumPy 的 ndarray，带有附加元数据：
np.asarray(y)

array([[-1.5],
       [ 0. ],
       [ 3.6],
       [ 1.3],
       [-2. ]])

In [73]:
np.asarray(X)

array([[ 1.  ,  1.  ,  0.01],
       [ 1.  ,  2.  , -0.01],
       [ 1.  ,  3.  ,  0.25],
       [ 1.  ,  4.  , -4.1 ],
       [ 1.  ,  5.  ,  0.  ]])

Intercept（截距）是从何而来？这是线性模型（比如普通最小二乘回归）的惯例用法。添加 +0 到模型可以不显示截距：

In [80]:
# 矩阵有两个，一个是y因变量矩阵，另一个就是x0 和 x1的组合的矩阵
# [1] 表示取出元组中的第二个元素，也就是设计矩阵，即包含自变量 x0 和 x1 的矩阵。
patsy.dmatrices("y ~ x0 + x1 + 0", data)[1]  

DesignMatrix with shape (5, 2)
  x0     x1
   1   0.01
   2  -0.01
   3   0.25
   4  -4.10
   5   0.00
  Terms:
    'x0' (column 0)
    'x1' (column 1)

此时，全为1的截距都没了

Patsy对象可以直接传递给算法，比如numpy.linalg.lstsq，它执行普通最小二乘回归：

In [47]:
coef, resid, _, _ = np.linalg.lstsq(X, y)

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_19728\2525922789.py:1: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  coef, resid, _, _ = np.linalg.lstsq(X, y)


报错只是需要添加新的参数

In [88]:
# 利用最小二乘法拟合线性模型，计算 X 和 y 之间的最佳线性关系
# 并得到模型的系数（coef）和残差（resid）。
coef, resid, _, _ = np.linalg.lstsq(X, y, rcond=-1)  # 或rcond=None

np.linalg.lstsq() 是一个函数，用于解决最小二乘问题，公式形式为：  

![jupyter](12.1.png)  

它会找到 coef（系数矩阵），使得线性模型中 X @ coef 最接近 y。这通常用于线性回归问题中，来拟合自变量和因变量之间的关系。



**参数：**  
X：这是设计矩阵或特征矩阵，形状为 (n_samples, n_features)，表示 n_samples 行样本，n_features 列特征。  
y：这是因变量或目标变量的矩阵，形状为 (n_samples, ) 或 (n_samples, 1)。  



**返回值：**  
**coef：**回归系数，表示拟合的线性模型中的系数。即使 X 矩阵乘以这个系数时，能够最好地逼近 y。  
**resid：**残差，即实际值与预测值之间的误差。如果 X 的行数大于列数（即方程组超定），会返回一个残差值；否则返回空值。  
**_：**表示奇异值，通常可以忽略。  
**_：**最后一个值是矩阵的秩，但这里没有被使用。  

模型的元数据保留在design_info属性中，因此你可以将模型列名重新附加到拟合系数上，以获得一个Series

In [93]:
# 回归系数
coef

array([[ 0.31290976],
       [-0.07910564],
       [-0.26546384]])

In [95]:
# 将最小二乘法回归得到的 coef（系数数组）转换为一个带有特征名称索引的 Pandas 系列（Series）。
# 每个系数会对应一个特征名称，方便进行进一步的数据分析和解读。
coef = pd.Series(coef.squeeze(), index=X.design_info.column_names)

coef

Intercept    0.312910
x0          -0.079106
x1          -0.265464
dtype: float64

行索引分别是：截距、x0、x1

#### 12.2.1 用Patsy公式进行数据转换 

你可以将Python代码与patsy公式结合。在执行公式时，Patsy库将尝试在封闭作用域内查找使用的函数：

In [101]:
y, X = patsy.dmatrices("y ~ x0 +np.log(np.abs(x1) + 1)", data)
X

DesignMatrix with shape (5, 3)
  Intercept  x0  np.log(np.abs(x1) + 1)
          1   1                 0.00995
          1   2                 0.00995
          1   3                 0.22314
          1   4                 1.62924
          1   5                 0.00000
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'np.log(np.abs(x1) + 1)' (column 2)

常见的变量转换包括标准化（均值为0，方差为1）和居中（减去平均值）。  
Patsy有内置的函数进行此工作：

In [104]:
y, X = patsy.dmatrices("y ~ standardize(x0) + center(x1)", data)
X

DesignMatrix with shape (5, 3)
  Intercept  standardize(x0)  center(x1)
          1         -1.41421        0.78
          1         -0.70711        0.76
          1          0.00000        1.02
          1          0.70711       -3.33
          1          1.41421        0.77
  Terms:
    'Intercept' (column 0)
    'standardize(x0)' (column 1)
    'center(x1)' (column 2)

作为建模过程的一环，你可能将模型拟合到一个数据集，然后用另一个数据集来评估模型。  
另一个数据集可能是剩余的部分或是新数据。  
当执行居中和标准化等转换时，使用模型对新数据进行预测要格外小心。  
因为必须使用原始数据集的平均值或标准差等统计值来对新数据集做转换，所以也称作有状态转换。

patsy.build_design_matrices函数可以使用原始样本数据集的保存信息来转换样本外的新数据：

In [108]:
new_data = pd.DataFrame({"x0": [6, 7, 8, 9],
                         'x1': [3.1, -0.5, 0, 2.3],
                         'y': [1, 2, 3, 4]})

new_X = patsy.build_design_matrices([X.design_info], new_data)

new_X

[DesignMatrix with shape (4, 3)
   Intercept  standardize(x0)  center(x1)
           1          2.12132        3.87
           1          2.82843        0.27
           1          3.53553        0.77
           1          4.24264        3.07
   Terms:
     'Intercept' (column 0)
     'standardize(x0)' (column 1)
     'center(x1)' (column 2)]

加号（+）在Patsy的上下文中不表示加法，当你按照名称将数据集的列相加时，必须用特殊的函数 I 将列名封装起来：

In [111]:
y, X = patsy.dmatrices('y ~ I(x0 + x1)', data)

X

DesignMatrix with shape (5, 2)
  Intercept  I(x0 + x1)
          1        1.01
          1        1.99
          1        3.25
          1       -0.10
          1        5.00
  Terms:
    'Intercept' (column 0)
    'I(x0 + x1)' (column 1)

Patsy的patsy.builtins模块中还有一些其他的内置转换

#### 12.2.2 分类数据和Patsy  
可以用多种方式将非数值数据转换为模型设计矩阵，当你在Patsy公式中使用非数值数据时，会默认将其转换为虚拟变量。如果有截距，会去掉其中一个，以避免共线性：

In [116]:
data = pd.DataFrame({'key1': ['a', 'a', 'b', 'b', 'a', 'b', 'a', 'b'],
                     'key2': [0, 1, 0, 1, 0, 1, 0, 0],
                     'v1': [1, 2, 3, 4, 5, 6, 7, 8],
                     'v2': [-1, 0, 2.5, -0.5, 4.0, -1.2, 0.2, -1.7]})

y, X = patsy.dmatrices('v2 ~ key1', data)
X

DesignMatrix with shape (8, 2)
  Intercept  key1[T.b]
          1          0
          1          0
          1          1
          1          1
          1          0
          1          1
          1          0
          1          1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)

如果你从模型中忽略截距，则每个分类值的列都会包含在模型设计矩阵中：

In [119]:
y, X = patsy.dmatrices('v2 ~ key1 + 0', data)
X

DesignMatrix with shape (8, 2)
  key1[a]  key1[b]
        1        0
        1        0
        0        1
        0        1
        1        0
        0        1
        1        0
        0        1
  Terms:
    'key1' (columns 0:2)

使用C函数，可以将数值列解释为分类类型：

In [122]:
y, X = patsy.dmatrices('v2 ~ C(key2)', data)
X

DesignMatrix with shape (8, 2)
  Intercept  C(key2)[T.1]
          1             0
          1             1
          1             0
          1             1
          1             0
          1             1
          1             0
          1             0
  Terms:
    'Intercept' (column 0)
    'C(key2)' (column 1)

当你在模型中使用多个分类项时，事情就会变复杂，因为会包括key1：key2形式的交互项，它可以用在方差分析（ANOVA）模型中：

In [125]:
data['key2'] = data['key2'].map({0: 'zero', 1: 'one'})
data

,key1,key2,v1,v2
0,a,zero,1,-1.0
1,a,one,2,0.0
2,b,zero,3,2.5
3,b,one,4,-0.5
4,a,zero,5,4.0
5,b,one,6,-1.2
6,a,zero,7,0.2
7,b,zero,8,-1.7


In [127]:
y, X = patsy.dmatrices('v2 ~ key1 + key2', data)
X

DesignMatrix with shape (8, 3)
  Intercept  key1[T.b]  key2[T.zero]
          1          0             1
          1          0             0
          1          1             1
          1          1             0
          1          0             1
          1          1             0
          1          0             1
          1          1             1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)

In [129]:
y, X = patsy.dmatrices('v2 ~ key1 + key2 + key1:key2', data)
X

DesignMatrix with shape (8, 4)
  Intercept  key1[T.b]  key2[T.zero]  key1[T.b]:key2[T.zero]
          1          0             1                       0
          1          0             0                       0
          1          1             1                       1
          1          1             0                       0
          1          0             1                       0
          1          1             0                       0
          1          0             1                       0
          1          1             1                       1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)
    'key1:key2' (column 3)

Patsy中还有转换分类数据的其他方法，包括以特定顺序进行转换。

### 12.3 statsmodels 介绍  
statsmodels（**https://www.statsmodels.org**）是Python用于拟合多种统计模型、进行统计试验、数据探索和可视化的库。  
statsmodels包含许多经典的频率论统计方法，可以在其他库中找到贝叶斯方法和机器学习模型。  

**statsmodels包含以下模型：**  
线性模型，包括广义线性模型和鲁棒线性模型  
线性混合效应模型  
方差分析方法  
时间序列过程和状态空间模型  
广义矩估计  

#### 12.3.1 对线性模型进行估计  
statsmodels有多种线性回归模型，包括从基本（例如，普通最小二乘）到复杂（例如，选代加权最小二乘）的模型。  
statsmodels的线性模型有两种不同的接口：基于数组和基于公式。
它们可以通过对应的API模块导人来访问：  

In [136]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [144]:
# 随机数生成线性模型：
rng = np.random.default_rng(seed=12345)

# 辅助函数生成特定均值和方差的正态分布数据
def dnorm(mean, variance, size=1):
    if isinstance(size, int):
        size = size
    return mean + np.sqrt(variance) * rng.standard_normal(size)

N = 100
X = np.c_[dnorm(0, 0.4, size=N),
          dnorm(0, 0.6, size=N),
          dnorm(0, 0.2, size=N)]


eps = dnorm(0, 0.1, size=N)

# 已知参数
beta = [0.1, 0.3, 0.5]

y = np.dot(X, beta) + eps

这里，我使用了“真实”模型和已知参数beta。  
此时，dnor是辅助函数，用于生成具有特定的均值和方差的正态分布数据。

In [147]:
X[:5]

array([[-0.90050602, -0.18942958, -1.0278702 ],
       [ 0.79925205, -1.54598388, -0.32739708],
       [-0.55065483, -0.12025429,  0.32935899],
       [-0.16391555,  0.82403985,  0.20827485],
       [-0.04765129, -0.21314698, -0.04824364]])

In [149]:
y[:5]

array([-0.59952668, -0.58845445,  0.18563386, -0.00747657, -0.01537445])

像之前在Patsy中看到的，线性模型通常要和--个截距项拟合。sm.add_constant函数可以添加一个截距列到现有的矩阵：

In [152]:
X_model = sm.add_constant(X)
X_model[:5]

array([[ 1.        , -0.90050602, -0.18942958, -1.0278702 ],
       [ 1.        ,  0.79925205, -1.54598388, -0.32739708],
       [ 1.        , -0.55065483, -0.12025429,  0.32935899],
       [ 1.        , -0.16391555,  0.82403985,  0.20827485],
       [ 1.        , -0.04765129, -0.21314698, -0.04824364]])

In [156]:
# sm.OLS类可以拟合普通最小二乘回归：
model = sm.OLS(y, X)

# 这个模型的fit方法返回了一个回归结果对象，它包含估计的模型参数和其他诊断信息：
results = model.fit()

results.params

array([0.06681503, 0.26803235, 0.45052319])

In [160]:
# 使用summary方法可以打印模型的详细诊断结果：
print(results.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.469
Model:                            OLS   Adj. R-squared (uncentered):              0.452
Method:                 Least Squares   F-statistic:                              28.51
Date:                Fri, 04 Oct 2024   Prob (F-statistic):                    2.66e-13
Time:                        16:34:01   Log-Likelihood:                         -25.611
No. Observations:                 100   AIC:                                      57.22
Df Residuals:                      97   BIC:                                      65.04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

这里的参数名为通用名x1、x2等。

In [163]:
# 假设所有的模型参数都在一个DF中：
data = pd.DataFrame(X, columns=['col0', 'col1', 'col2'])
data['y'] = y

data[:5]

,col0,col1,col2,y
0,-0.900506,-0.189430,-1.027870,-0.599527
1,0.799252,-1.545984,-0.327397,-0.588454
2,-0.550655,-0.120254,0.329359,0.185634
3,-0.163916,0.824040,0.208275,-0.007477
4,-0.047651,-0.213147,-0.048244,-0.015374


In [165]:
# 现在使用statsmodel的公式API和Patsy的公式字符串：
results = smf.ols('y ~ col0 + col1 + col2', data=data).fit()

results.params

Intercept   -0.020799
col0         0.065813
col1         0.268970
col2         0.449419
dtype: float64

In [167]:
results.tvalues

Intercept   -0.652501
col0         1.219768
col1         6.312369
col2         6.567428
dtype: float64

观察statsmodels是如何返回Series结果的，它附带DataFrame的列名。当使用公式和pandas对象时，我们不需要使用add_constant。

In [171]:
# 给出一个样本外数据，你可以根据估计的模型参数计算预测值：
results.predict(data[:5])

0   -0.592959
1   -0.531160
2    0.058636
3    0.283658
4   -0.102947
dtype: float64

statsmodels中还有很多其他工具，可以对线性模型结果进行分析、诊断和可视化。除了
普通最小二乘，statsmodels中还有其他线性模型。

#### 12.3.2 对时间序列过程进行估计  
statsmodels的另一类模型是对时间序列进行分析，包括自回归过程、卡尔曼滤波和其他状态空间模型，以及多变量自回归模型。

In [187]:
# 用自回归结构和噪声模拟一些时间序列数据：
init_x = 4
values = [init_x, init_x]  # 初始值
N = 1000  # 生成的数据点数

b0 = 0.8  # 第一个滞后的系数
b1 = -0.4  # 第二个滞后的系数
noise = dnorm(0, 0.1, N)  # 噪声项，服从均值为 0 方差为 0.1 的正态分布

for i in range(N):
    new_x = values[-1] * b0 + values[-2] * b1 + noise[i]  # 自回归结构
    values.append(new_x)  # 将新的值添加到时间序列中

# 拟合自回归模型
from statsmodels.tsa.ar_model import AutoReg

MAXLAGS = 5  # 最大滞后阶数为 5
model = AutoReg(values, MAXLAGS)  # 使用最大滞后为 5 的自回归模型
results = model.fit()  # 拟合模型

# 结果中的估计参数首先是截距，其次是前两个滞后的估计：
results.params

array([-0.0099209 ,  0.76293185, -0.3501795 ,  0.00465461, -0.00625459,
        0.02586898])

这个数据具有AR(2)结构（两个滞后），参数是0.8和-0.4。当拟合AR模型时，你可能不知道要包含的滞后项的个数，因此可以用更大的滞后数来拟合这个模型：

### 12.4 scikit-learn介绍  
scikit-learn（**https://scikit-learn.org**）是使用广泛、用途多样的Python机器学习库之一。  
它包含多种标准的监督机器学习和非监督机器学习方法，以及模型选择和评估、数据转换、数据加载和模型持久化工具。这些模型可以用于分类、聚类、预测和其他常见任务。

In [193]:
# 计算泰坦尼克号的乘客生还率：
train = pd.read_csv('../datasets/titanic/train.csv')
test = pd.read_csv('../datasets/titanic/test.csv')
train.head(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


**statsmodels和scikit-learn通常不能接收缺失数据，因此我们要查看列是否包含缺失值：**

In [197]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [199]:
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

在像这样的统计和机器学习示例中，根据数据中的特征，一个典型的任务是预测乘客能否生还。  
模型先在训练数据集中拟合，然后用样本外测试数据集进行评估。

用Age作为预测值，但是它包含缺失值。有多种补全缺失数据的方法，现在用简单方法，用训练数据集的中位数补全两个表的空值：

In [213]:
impute_value = train['Age'].median()

train['Age'] = train['Age'].fillna(impute_value)

test['Age'] = test['Age'].fillna(impute_value)

# 指定模型训练集测试集，增添一列IsFemale作为‘Sex’列的编码：
train['IsFemale'] = (train['Sex'] == 'female').astype(int)
test['IsFemale'] = (test['Sex'] == 'female').astype(int)

# 确定模型变量，创建Numpy数组：
predictors = ['Pclass', 'IsFemale', 'Age']
X_train = train[predictors].to_numpy()
X_test = test[predictors].to_numpy()
y_train = train['Survived'].to_numpy()

X_train[:5]

array([[ 3.,  0., 22.],
       [ 1.,  1., 38.],
       [ 3.,  1., 26.],
       [ 1.,  1., 35.],
       [ 3.,  0., 35.]])

In [209]:
y_train[:5]

array([0, 1, 1, 1, 0], dtype=int64)

不能保证这是一个好模型，也不能保证这些特征得到了合适的处理。我们使用scikit-learn的LogisticRegression模型创建一个模型实例：

In [215]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

# 使用模型的fit方法，将模型拟合到训练数据：
model.fit(X_train, y_train)

LogisticRegression()

In [217]:
# 现在可以用model.predict在测试集上进行预测：
y_predict = model.predict(X_test)

y_predict[:10]

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0], dtype=int64)

In [ ]:
# 如果有测试数据集的真实值，可以计算准确率或其他误差度指标
(y_true == y_predict).mean()

在实际中，模型训练经常有许多额外的复杂因素。许多模型有可以调节的参数，有些方法（比如交叉验证）可以用来进行参数调节，避免对训练数据过拟合。这通常可以对新数据提高预测表现或健壮性。

交叉验证通过分割训练数据来模拟样本外预测。基于模型的准确度分数（比如均方差），可以对模型参数进行网格搜索。  
有些模型，如logistic回归，有内置的交叉验证的估计类。  
例如，LogisticRegressionCV类可以用一个参数指定对模型正则化参数C的网格搜索粒度：

In [222]:
from sklearn.linear_model import LogisticRegressionCV

model_cv = LogisticRegressionCV(Cs=10)

model_cv.fit(X_train, y_train)

LogisticRegressionCV()

要手动进行交叉验证，你可以使用辅助函数cross_val_score，它可以处理数据分割过程。

In [225]:
# 要使用训练数据的4个非重叠部分来交叉验证我们的模型：
from sklearn.model_selection import cross_val_score

model = LogisticRegression(C=10)

scores = cross_val_score(model, X_train, y_train, cv=4)

scores

array([0.77578475, 0.79820628, 0.77578475, 0.78828829])

默认的评分指标取决于模型本身，但是可以显式指定一个评分函数。交叉验证过的模型需要更长时间来训练，但会有更好的模型性能。